# Step 04: Identify first and last corresponding author affiliation

Input:
    data/interim/publication/2_first_combined_predictresult.csv
    data/interim/publication/2_last_combined_predictresult.csv

Output:
    data/interim/publication/3_first_gender_affiliation.csv
    data/interim/publication/3_last_gender_affiliation.csv


Purpose:
    - Identify the affiliation of authors

In [ ]:
import pandas as pd
import re
from rapidfuzz import fuzz, process
from tqdm import tqdm
import numpy as np
from pathlib import Path

# 1 First author

In [ ]:
# -----------------------------
# Paths
# -----------------------------

PROJECT_ROOT = Path(__file__).resolve().parents[2]

FIRST_GENDER_FILE = PROJECT_ROOT / "data" / "interim" / "publication"/ "2.1_first_NameAndCountry_Predicted.csv"
FIRST_INS_FILE = PROJECT_ROOT / "data" / "interim" / "publication"/ "2.0_first_NameAndCountry.csv"


FIRST_OUT_PATH = PROJECT_ROOT / "data" / "interim" / "publication"/ "3_first_gender_affiliation.csv"
FIRST_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
first = pd.read_csv(FIRST_GENDER_FILE)
print(first.columns)
first

ins = pd.read_csv(FIRST_INS_FILE)
ins

In [ ]:
# Affiliation based on the address information

def extract_abbreviation(address):
    """
    Extract the abbreviation (text between first and second comma) from address.
    """
    if pd.isna(address) or not isinstance(address, str):
        return ""
    
    # Split by comma and get the second element (index 1)
    parts = address.split(',')
    if len(parts) >= 2:
        return parts[1].strip()
    return ""

def preprocess_text(text):
    """
    Preprocess text for better matching by removing common words and normalizing.
    """
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase and remove extra whitespace
    text = re.sub(r'\s+', ' ', text.lower().strip())
    
    # Remove common words that might interfere with matching
    common_words = ['university', 'univ', 'institute', 'inst', 'college', 'school', 
                   'center', 'centre', 'dept', 'department', 'lab', 'laboratory',
                   'medical', 'med', 'technology', 'tech', 'of', 'the', 'and']
    
    # Keep original for initial matching, but create a cleaned version for fallback
    words = text.split()
    cleaned_words = [word for word in words if word not in common_words]
    
    return text, ' '.join(cleaned_words) if cleaned_words else text

def find_best_match(abbreviation, affiliations_list, threshold=60):
    """
    Find the best matching affiliation using fuzzy matching.
    Uses multiple matching strategies for better accuracy.
    """
    if not abbreviation or not affiliations_list:
        return affiliations_list[0] if affiliations_list else ""
    
    # If only one affiliation, return it
    if len(affiliations_list) == 1:
        return affiliations_list[0]
    
    # Preprocess abbreviation
    abbr_original, abbr_cleaned = preprocess_text(abbreviation)
    
    best_match = ""
    best_score = 0
    
    for affiliation in affiliations_list:
        if not isinstance(affiliation, str):
            continue
            
        # Preprocess affiliation
        aff_original, aff_cleaned = preprocess_text(affiliation)
        
        # Strategy 1: Direct partial ratio on original text
        score1 = fuzz.partial_ratio(abbr_original, aff_original)
        
        # Strategy 2: Token sort ratio on original text
        score2 = fuzz.token_sort_ratio(abbr_original, aff_original)
        
        # Strategy 3: Partial ratio on cleaned text (removes common words)
        score3 = fuzz.partial_ratio(abbr_cleaned, aff_cleaned)
        
        # Strategy 4: Check if abbreviation words are contained in affiliation
        abbr_words = set(abbr_original.split())
        aff_words = set(aff_original.split())
        word_overlap = len(abbr_words.intersection(aff_words)) / len(abbr_words) * 100 if abbr_words else 0
        
        # Combine strategies with weights
        combined_score = max(
            score1 * 0.3 + score2 * 0.3 + score3 * 0.2 + word_overlap * 0.2,
            score1,  # Sometimes direct partial ratio is best
            word_overlap  # Sometimes word overlap is most reliable
        )
        
        if combined_score > best_score and combined_score >= threshold:
            best_score = combined_score
            best_match = affiliation
    
    # If no good match found, return the first affiliation or empty string
    return best_match if best_match else (affiliations_list[0] if affiliations_list else "")

def add_affiliation_confirm(df, batch_size=1000):
    """
    Add 'affiliation_confirm' column to dataframe with progress bar.
    Processes in batches for memory efficiency.
    """
    # Initialize the new column
    df['affiliation_confirm'] = ""
    
    # Process in batches to save memory and show progress
    total_rows = len(df)
    
    with tqdm(total=total_rows, desc="Processing affiliations", unit="rows") as pbar:
        for start_idx in range(0, total_rows, batch_size):
            end_idx = min(start_idx + batch_size, total_rows)
            batch_indices = range(start_idx, end_idx)
            
            for idx in batch_indices:
                # Extract abbreviation from address
                abbreviation = extract_abbreviation(df.loc[idx, 'address'])
                
                # Parse affiliations list
                affiliations_str = df.loc[idx, 'affiliations']
                if pd.isna(affiliations_str):
                    affiliations_list = []
                else:
                    affiliations_list = [aff.strip() for aff in str(affiliations_str).split(';') if aff.strip()]
                
                # Find best match
                best_match = find_best_match(abbreviation, affiliations_list)
                df.loc[idx, 'affiliation_confirm'] = best_match
                
                pbar.update(1)
    
    return df

In [ ]:
ins_confirm = add_affiliation_confirm(ins)


ins_confirm = ins_confirm.rename(columns={'affiliation_confirm': 'affiliation'})
ins_confirm['affiliation'] = ins_confirm['affiliation'].replace("", np.nan)
ins_confirm = ins_confirm.drop(['author','number of authors','address','country1','country2','country'], axis=1)
ins_confirm

In [ ]:
# Merge with predicted gender
first = first.merge(ins_confirm, on='index_num', how='left')
first


first.to_csv(FIRST_OUT_PATH, index=False)

# 2 Last Correpsonding author

In [ ]:
# -----------------------------
# Paths
# -----------------------------

PROJECT_ROOT = Path(__file__).resolve().parents[2]

ALL_PUB_FILE = PROJECT_ROOT / "data" / "interim" / "publication"/ "1_AllPublications.csv"

LAST_GENDER_FILE = PROJECT_ROOT / "data" / "interim" / "publication"/ "2.1_last_NameAndCountry_Predicted.csv"
LAST_INS_FILE = PROJECT_ROOT / "data" / "interim" / "publication"/ "2.0_last_NameAndCountry.csv"


LAST_OUT_PATH = PROJECT_ROOT / "data" / "interim" / "publication"/ "3_last_gender_affiliation.csv"
LAST_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
file_path = LAST_GENDER_FILE
name_gend_idx = pd.read_csv(file_path)
name_gend_idx = name_gend_idx.sort_values(by='index_num')

file_path = LAST_INS_FILE
name_addr_idx_ctry = pd.read_csv(file_path)
name_addr_idx_ctry = name_addr_idx_ctry.sort_values(by='index_num')

name_gend_idx


cols_to_import = ['index_num', 'Affiliations']
idx_Affl = pd.read_csv(ALL_PUB_FILE, usecols=cols_to_import)


In [ ]:
#merge all information
nam_idx_ctr_gen_adr=pd.merge(name_gend_idx, name_addr_idx_ctry, how="left", on=["author","index_num"])
nam_idx_ctr_gen_adr = nam_idx_ctr_gen_adr.drop(columns=['country_y'])
nam_idx_ctr_gen_adr = nam_idx_ctr_gen_adr.rename(columns={'country_x': 'country'})
nam_idx_ctr_gen_adr

nam_idx_ctr_gen_adr_afl = pd.merge(nam_idx_ctr_gen_adr, idx_Affl, how="left", on=["index_num"])

In [ ]:
# Identify Affiliation based on the address information
def extract_abbreviation(address):
    """
    Extract the abbreviation (text between first and second comma) from address.
    """
    if pd.isna(address) or not isinstance(address, str):
        return ""
    
    # Split by comma and get the second element (index 1)
    parts = address.split(',')
    if len(parts) >= 2:
        return parts[1].strip()
    return ""

def preprocess_text(text):
    """
    Preprocess text for better matching by removing common words and normalizing.
    """
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase and remove extra whitespace
    text = re.sub(r'\s+', ' ', text.lower().strip())
    
    # Remove common words that might interfere with matching
    common_words = ['university', 'univ', 'institute', 'inst', 'college', 'school', 
                   'center', 'centre', 'dept', 'department', 'lab', 'laboratory',
                   'medical', 'med', 'technology', 'tech', 'of', 'the', 'and']
    
    # Keep original for initial matching, but create a cleaned version for fallback
    words = text.split()
    cleaned_words = [word for word in words if word not in common_words]
    
    return text, ' '.join(cleaned_words) if cleaned_words else text

def find_best_match(abbreviation, affiliations_list, threshold=60):
    """
    Find the best matching affiliation using fuzzy matching.
    Uses multiple matching strategies for better accuracy.
    """
    if not abbreviation or not affiliations_list:
        return affiliations_list[0] if affiliations_list else ""
    
    # If only one affiliation, return it
    if len(affiliations_list) == 1:
        return affiliations_list[0]
    
    # Preprocess abbreviation
    abbr_original, abbr_cleaned = preprocess_text(abbreviation)
    
    best_match = ""
    best_score = 0
    
    for affiliation in affiliations_list:
        if not isinstance(affiliation, str):
            continue
            
        # Preprocess affiliation
        aff_original, aff_cleaned = preprocess_text(affiliation)
        
        # Strategy 1: Direct partial ratio on original text
        score1 = fuzz.partial_ratio(abbr_original, aff_original)
        
        # Strategy 2: Token sort ratio on original text
        score2 = fuzz.token_sort_ratio(abbr_original, aff_original)
        
        # Strategy 3: Partial ratio on cleaned text (removes common words)
        score3 = fuzz.partial_ratio(abbr_cleaned, aff_cleaned)
        
        # Strategy 4: Check if abbreviation words are contained in affiliation
        abbr_words = set(abbr_original.split())
        aff_words = set(aff_original.split())
        word_overlap = len(abbr_words.intersection(aff_words)) / len(abbr_words) * 100 if abbr_words else 0
        
        # Combine strategies with weights
        combined_score = max(
            score1 * 0.3 + score2 * 0.3 + score3 * 0.2 + word_overlap * 0.2,
            score1,  # Sometimes direct partial ratio is best
            word_overlap  # Sometimes word overlap is most reliable
        )
        
        if combined_score > best_score and combined_score >= threshold:
            best_score = combined_score
            best_match = affiliation
    
    # If no good match found, return the first affiliation or empty string
    return best_match if best_match else (affiliations_list[0] if affiliations_list else "")

def add_affiliation_confirm(df, batch_size=1000):
    """
    Add 'affiliation_confirm' column to dataframe with progress bar.
    Processes in batches for memory efficiency.
    """
    # Initialize the new column
    df['affiliation_confirm'] = ""
    
    # Process in batches to save memory and show progress
    total_rows = len(df)
    
    with tqdm(total=total_rows, desc="Processing affiliations", unit="rows") as pbar:
        for start_idx in range(0, total_rows, batch_size):
            end_idx = min(start_idx + batch_size, total_rows)
            batch_indices = range(start_idx, end_idx)
            
            for idx in batch_indices:
                # Extract abbreviation from address
                abbreviation = extract_abbreviation(df.loc[idx, 'address'])
                
                # Parse affiliations list
                affiliations_str = df.loc[idx, 'Affiliations']
                if pd.isna(affiliations_str):
                    affiliations_list = []
                else:
                    affiliations_list = [aff.strip() for aff in str(affiliations_str).split(';') if aff.strip()]
                
                # Find best match
                best_match = find_best_match(abbreviation, affiliations_list)
                df.loc[idx, 'affiliation_confirm'] = best_match
                
                pbar.update(1)
    
    return df

In [ ]:
df = add_affiliation_confirm(nam_idx_ctr_gen_adr_afl)

In [ ]:
df = df.drop(columns=['Affiliations'])
df = df.rename(columns={'affiliation_confirm': 'affiliation'})
df['affiliation'] = df['affiliation'].replace("", np.nan)
df


df_dedup = df[~df['index_num'].isin(df['index_num'][df['index_num'].duplicated(keep=False)])]
df_dedup

In [ ]:
# Keep only one record per publication - last corresponding author

full_duplicates = df[df['index_num'].duplicated(keep=False)]
print(full_duplicates)

use_col = ['index_num','Author Full Names']
ref = pd.read_csv(ALL_PUB_FILE, usecols=use_col)   #refer to the full publication records

merged = full_duplicates.merge(ref, on='index_num', how='left')
merged

In [ ]:
# Function to find the rank of an author in the author full names list

def get_author_rank(author, author_full_names):
    """
    Find the rank (1-indexed) of an author in the author full names list
    Returns the position of the author, or None if not found
    """
    if pd.isna(author) or pd.isna(author_full_names):
        return None
    
    # Normalize function to remove whitespace and periods
    def normalize_name(name):
        return str(name).strip().replace('.', '')
    
    # Split the author full names by semicolon and normalize each name
    authors_list = [normalize_name(name) for name in str(author_full_names).split(';')]
    normalized_author = normalize_name(author)
    
    try:
        # Find the index (0-based) and convert to 1-based rank
        rank = authors_list.index(normalized_author) + 1
        return rank
    except ValueError:
        # Author not found in the list
        return None

# Create the 'rank' column
merged['rank'] = merged.apply(lambda row: get_author_rank(row['author'], row['Author Full Names']), axis=1)

# For each index_num, find the maximum rank and mark those rows as 1
merged['last'] = (merged['rank'] == merged.groupby('index_num')['rank'].transform('max')).astype(int)
print(merged.head(20))

# keep only rows where last == 1
last_rows = merged[merged['last'] == 1].copy()

# drop the helper columns
last_rows = last_rows.drop(columns=['rank', 'last','Author Full Names'])



In [ ]:
# append back with non duplicated records
df_dedup = pd.concat([df_dedup, last_rows], ignore_index=True)

In [ ]:
df_dedup.to_csv(LAST_OUT_PATH, index=False)